<img src="../logo.png" alt="Vegeta" width="240">

# Kestrel-Q4T — the tandem-wing Kestrel: Q4T1N-10 against Q4T1P-10, Q4T1N-12 and Kestrel-4Q-2

**Kestrel-Q4T** puts Kestrel's four motors, battery and pod on two equal wings one behind the other, no bracing: a
tandem wing that **starts horizontally** (a ground roll on skids, or a hand launch), **can pull up into a hover**, and
may **fall** on the way back to horizontal flight. The design brief with the first principles is
`designs/kestrel_tandem.md`; this notebook is the study of the airframe the brief selected and its comparisons:

| | arrangement | what it answers |
|---|---|---|
| **Q4T1N-10** | front wing low, rear wing high (Quickie), span 1.0 m | **the design** |
| Q4T1P-10 | the mirror: front wing high | the stagger sign — the rear wing (the elevator) in or out of the front's wake |
| Q4T1N-12 | the design at span 1.2 m | what the span a horizontal start frees is worth |
| Q4T1-10 (flat) | both wings in one plane | the control case: Munk's +40 % induced drag, no pitch lever in the hover |
| Kestrel-4Q-2 | the biplane, from notebook 43's report | the previous design |

Method labels as the other Kestrel notebooks: the lattice and the beam rules are *solver* (inviscid, linear), the
drag build-up and Prandtl's σ *empirical*, the drive points *Boreas*; nothing here is a RANS or FEA result. Under the
15–30 min rule the notebook launches no solver; the CFD and FEA chapters of a book `kestrel-tandem` are the plan's
next step (`designs/kestrel_tandem.md` § 8).

In [ ]:
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)        # the lab's folders on sys.path, the mocks

In [ ]:
import dataclasses, json, math, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0, ".")
import kestrel, kestrel_systems as ks, kestrel_control as kc, kestrel_flight as kf, kestrel_transition as kt, kestrel_tandem as ktd, kestrel_drawings as kd, kestrel_xwing as kx
from vegeta import dedalus

pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
OUT = Path("../output/46_kestrel_tandem"); OUT.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("TANDEM_QUICK") == "1"
T0 = time.time()
import yanus                                       # Ozaru's requirement checks: what this notebook needs on this machine
from yanus import Package
NEEDS = [Package("vegeta.boreas"),
         Package("vegeta.dedalus")]
print(yanus.table(yanus.check_needs(NEEDS)) if NEEDS else "needs: nothing beyond the Python environment")
BIP = ktd.biplane_reference()                       # notebook 43's report (designs/data/reports/43_kestrel_study.json); None, with a loud warning, when it is missing
MAIN = ktd.MAIN
print("Kestrel-4Q-2 (the previous design):", f"notebook 43's report, {BIP['source']}" if BIP else "NOT FOUND — its rows are left out (see the warning)")

## 1. The four airframes

The layout from the brief: front wing's leading edge 200 mm behind the nose, stagger 3 chords, gap 0.28 m (the biplane's
hover lever and Prandtl σ), motors at ±0.35 m with tractor propellers 75 mm ahead of each wing, the low wing's motors
on 50 mm pylons so that a 10-inch propeller clears the ground on 100 mm skids, the pod spanning both wings, a small
fin on the rear wing. The rules: discs clear the pod and the tips, propellers clear the ground, the rear wing at
least 1.5 chords behind the front wing's trailing edge.

In [ ]:
LAY = {v: ktd.layout(v) for v in ktd.VARIANTS}
L = LAY[MAIN]
fig, axs = plt.subplots(1, 2, figsize=(15, 6))
ktd.front_view(LAY[MAIN], ax=axs[0]); ktd.front_view(LAY["Q4T1P-10"], ax=axs[1]); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(13, 4.5)); ktd.side_view(L, ax=ax); plt.show()
keys = ("span", "chord", "gap", "stagger", "x_le_front", "x_le_rear", "z_front", "z_rear", "S_ref", "AR_wing", "AR_total", "pod_length", "pod_height", "hover_pitch_lever", "prop_ground_clearance", "prop_to_pod", "elevon_y_in")
display(pd.DataFrame({v: {k: Lv[k] for k in keys} | {"arrangement": Lv["arrangement"], "rules": "; ".join(ktd.checks(Lv)) or "all hold"} for v, Lv in LAY.items()}).T)
design = dedalus.load_design("../../designs/kestrel_tandem.py:KestrelQ4T")
g = design.generate(**ktd.cad_params(MAIN), part="aircraft")
dedalus.plot_views(g, figsize=(13, 9)); plt.suptitle(f"Kestrel-{MAIN} (CAD: KestrelQ4T on the biplane's parts)", y=0.99); plt.show()
print(f"§ 1 in {time.time() - T0:.0f} s")

## 2. Masses, the centre of gravity and the spars

Kestrel's parts on this layout; the battery placed for the CG the tandem's trim wants — 10 % of the stagger ahead of
the pair's neutral point (§ 3) — and flagged when the pod's battery bay cannot reach it. Each half-wing is a
cantilever with its motor on one Ø10×8 carbon tube: the front one carries two thirds of a 4 g pull-up.

In [ ]:
TAB = {v: ktd.mass_table(Lv) for v, Lv in LAY.items()}
CI = {v: ks.cg_inertia(t) for v, t in TAB.items()}
display(pd.DataFrame({v: ks.groups(t) for v, t in TAB.items()}).round(3).rename_axis("mass groups [kg]"))
display(pd.DataFrame({v: {"take-off mass [kg]": c["mass_kg"], "CG x [mm]": c["x_cg"] * 1e3, "CG z [mm]": c["z_cg"] * 1e3, "battery x [mm]": c["battery_x"] * 1e3, "battery clipped": c["battery_clipped"],
                          "Ixx": c["Ixx"], "Iyy": c["Iyy"], "Izz": c["Izz"]} for v, c in CI.items()}).T.round(4))
SPAR = pd.DataFrame({v: ktd.cantilever(Lv, mass_kg=CI[v]["mass_kg"]) for v, Lv in LAY.items()}).T
display(SPAR.round(2))
if BIP:
    print("Kestrel-4Q, take-off mass (notebook 43):", {k: f"{v['take-off mass [kg]']:.3f} kg" for k, v in BIP["variants"].items()})

## 3. Aerodynamics: the lattice, Munk's theorem, the stall rule, the drag

The vortex lattice on the two staggered wings (the front with +2° of decalage): the lift slope, the induced-drag
factor, the neutral point, the front wing's share of the lift and each wing's effective slope against one wing alone
— the rear's is the downwash factor. **Munk's stagger theorem**: the induced drag of the system depends on the gap,
not on the stagger — the flat tandem pays what a single wing of the span pays, the staggered ones recover the
biplane's figure (Prandtl's (1+σ)/2 beside it). **The stall rule**: at trim the front wing carries
(x_ac,rear − x_cg) / stagger of the lift; it reaches the section's cl max first and the aircraft's CL max is the
section's × ½ / that share — the front-first stall that pitches the nose down.

In [ ]:
AER = {v: ktd.aero(Lv) for v, Lv in LAY.items()}
BIPL = kf.aero()["biplane"]
A = pd.DataFrame({v: {"CLα [1/rad]": a["CLa"], "CLα one wing": a["CLa_one_wing"], "k = CDi/CL²": a["k"], "e (span)": a["e"], "k / the biplane's": a["k"] / BIPL["k"],
                      "x_np from the front LE [c]": a["x_np_c_from_front_le"], "front share at α 4°": a["front_share_alpha4"], "front slope / alone": a["slope_front"],
                      "rear slope / alone (downwash)": a["slope_rear"], "CL at α 0 (decalage)": a["CL0"]} for v, a in AER.items()}).T
display(A.round(4).rename_axis("lattice [solver, inviscid]"))
MUNK = pd.Series({"biplane k (lattice, gap 0.28, no stagger)": BIPL["k"], "tandem flat k (lattice)": AER["Q4T1-10 (flat)"]["k"], "tandem N k (lattice)": AER[MAIN]["k"],
                  "flat / N (lattice)": AER["Q4T1-10 (flat)"]["k"] / AER[MAIN]["k"], "1 / ((1+σ)/2) at gap/span 0.28 (Prandtl)": 2 / (1 + kf.prandtl_sigma(0.28)),
                  "N / biplane (lattice)": AER[MAIN]["k"] / BIPL["k"]}, name="Munk: the gap decides the induced drag, not the stagger")
display(MUNK.round(3).to_frame().T)
STALL = pd.DataFrame({v: ktd.stall(Lv, AER[v], CI[v]["x_cg"]) for v, Lv in LAY.items()}).T
display(STALL.rename_axis("the stall rule [first principles]"))
DRAG = {v: ktd.drag_buildup(Lv) for v, Lv in LAY.items()}
display(pd.DataFrame({v: {k: x * 1e4 for k, x in d["parts_m2"].items()} | {"Cd0 on S_ref": d["Cd0"]} for v, d in DRAG.items()}).T.round(3).rename(columns=lambda c: c if c.startswith("Cd0") else c + " [cm²]").rename_axis("drag areas [empirical build-up]"))
POL = {v: ktd.polar(Lv, AER[v], DRAG[v], CI[v]["x_cg"]) for v, Lv in LAY.items()}
display(pd.DataFrame({v: {k: x for k, x in p.__dict__.items() if k != "source"} | {"α stall [deg]": math.degrees(p.alpha_stall)} for v, p in POL.items()}).T.round(4).rename_axis("the polars on S_ref"))
print(f"§ 3 in {time.time() - T0:.0f} s")

## 4. Trim, the elevator, the hover geometry

The rear pair of elevons is the elevator, 2.3 chords behind the CG; the front pair can be driven the other way. In
the hover the body x axis is vertical: a pitching moment by thrust needs propellers at different heights — the gap
(less the pylon) is the lever for N and P, nothing for the flat tandem, whose pitch control is the slipstream elevons
alone (the allocation matrix loses its rank). The same allocation and attainable-moment tables as the biplane's.

In [ ]:
ELEV = pd.DataFrame({v: ktd.elevator_power(Lv, AER[v], CI[v]["x_cg"]) for v, Lv in LAY.items()}).T
display(ELEV.round(3).rename_axis("elevator power about the CG [lattice slope × τ 0.45]"))
print("Kestrel-4Q's elevons for scale: all four at ~0.3 c behind the CG, Cm_δ ≈ −0.05 /rad together (notebook 43 § 6)")
DR = ks.drive_table()
HOV = {}
for v, Lv in LAY.items():
    ci = CI[v]
    print(v)
    display(kc.rank_table(kappa=float(DR.kap(0.0)), cg=ci, L=Lv)[["senses (M1..M4)", "rank", "condition", "verdict"]])
    HOV[v] = kc.hover_authority(mass_kg=ci["mass_kg"], cg=ci, inertia=ci["I"], L=Lv)
    display(HOV[v].round(3))

## 5. Level flight and the horizontal start

Each variant's drive (Boreas: the 10×7 on the 2216 at 4S, tabulated once and cached) at the drag of level flight, the
pitch trimmed by the rear elevator (its deflection and the lift it costs), the electrical power and the endurance on
the usable pack. The ground roll: full thrust from rest on the skids at 3° of incidence, rolling friction 0.08 (short
grass), to 1.15 × the stall speed.

In [ ]:
VEH = {v: ktd.make_vehicle(v) for v in ktd.VARIANTS}
CRU = {v: ktd.cruise(veh) for v, veh in VEH.items()}
for v, c in CRU.items():
    print(v); display(c.loc[[s for s in (11.0, 12.0, 14.0, 16.0, 20.0, 24.0) if s in c.index]].round(3))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for v, c in CRU.items():
    f = c[c["feasible"]]
    ax[0].plot(f.index, f["electrical [W]"], "o-", label=v); ax[1].plot(f.index, f["endurance [min]"], "o-", label=v)
for a_ in ax:
    a_.set_xlabel("airspeed [m/s]"); a_.legend(fontsize=8); a_.grid(alpha=0.3)
ax[0].set(ylabel="electrical [W]", title="level flight: power"); ax[1].set(ylabel="min", title="endurance on 80 % of the pack"); plt.show()
TO = pd.DataFrame({v: ktd.takeoff_roll(veh) for v, veh in VEH.items()}).T
display(TO.round(2).rename_axis("the horizontal start [point mass, full thrust]"))
print(f"§ 5 in {time.time() - T0:.0f} s")

## 6. The transitions — the pull-up into the hover, the way back out

Notebook 40's transition model with the tandem's four half-wings at their own stations (the rear ones with the
downwash factor, the front ones with the decalage, the elevator at its arm), each blown by its own rotor. Both
directions, three schedule lengths, two headwinds. The brief accepts a **fall** on the way out of the hover
(`forward`): the pass criterion on height loss (3 m) is the biplane's — read the height columns, not only `pass`.
The pull-up into the hover (`back`) is where a clean tandem gains height; the remedies the biplane study found
(a descent into the transition, a slower schedule) are tried on the design below.

In [ ]:
durs = (1.5, 3.0) if QUICK else (1.5, 3.0, 5.0)
MAT, EPS = kt.matrix(VEH, durations=durs, headwinds=(0.0, 6.0), progress=True)
cols = ["height loss [m]", "height gain [m]", "pitch error max [deg]", "time stalled, free stream [s]", "α blown max while stalled [deg]", "moment shortfall [s]",
        "least pitch margin", "peak rotor thrust / T_max", "elevon max [deg]", "roll demand / authority", "pass"]
display(MAT[cols].astype({c: float for c in cols if c != "pass"}).round(2))
VER = kt.verdict(MAT)
for k, v in VER.items():
    print(f"{k}: {v['passes']} of {v['of']} pass" + ("".join(f"\n   - {w}" for w in v["failures"]) if v["failures"] else ""))
MAT.to_csv(OUT / "transition_matrix.csv")
# the design's pull-up: what brings the balloon down
REM = pd.DataFrame({name: kt.simulate(VEH[MAIN], "back", t_tr=tt, **kw).metrics for name, tt, kw in (
    ("3 s schedule", 3.0, {}), ("5 s schedule", 5.0, {}), ("3 s, descent 3 m/s into it", 3.0, {"descent_entry": 3.0}), ("5 s, descent 3 m/s into it", 5.0, {"descent_entry": 3.0}),
    ("3 s, entered at 12 m/s", 3.0, {"V_entry": 12.0}))}).T
display(REM[cols].astype({c: float for c in cols if c != "pass"}).round(2).rename_axis(f"{MAIN}: the pull-up into the hover, remedies"))
FALL = pd.DataFrame({f"{v}, {tt} s": kt.simulate(veh, "forward", t_tr=tt).metrics for v, veh in VEH.items() for tt in ((3.0,) if QUICK else (1.5, 3.0))}).T
display(FALL[["height loss [m]", "height gain [m]", "speed at the end [m/s]", "time stalled, free stream [s]", "pitch error max [deg]", "pass"]].round(2).rename_axis("the way out of the hover: the fall"))
kd.episode_figure(EPS[(MAIN, "forward", 3.0, 0.0)], VEH[MAIN]); plt.show()
kd.episode_figure(EPS[(MAIN, "back", 3.0, 0.0)], VEH[MAIN]); plt.show()
print(f"§ 6 in {time.time() - T0:.0f} s")

## 7. Against Kestrel-4Q-2 and between the variants

The variants side by side, Kestrel-4Q-2 from notebook 43's report in the same rows where it has them; then the drag,
power and maximum speed against airspeed for the design, its two alternatives and the biplane rebuilt from the same
modules (`kestrel_transition.make_vehicle`, elevons and faired posts).

In [ ]:
rows = {}
for v in ktd.VARIANTS:
    Lv, veh, c, st = LAY[v], VEH[v], CRU[v], STALL.loc[v]
    f = c[c["feasible"]]; c16 = c.loc[16.0] if 16.0 in c.index else None
    hx = HOV[v]
    rows[v] = {"mass [kg]": veh.mass, "wing area [m²]": Lv["S_ref"], "span [m]": Lv["span"], "CLα [1/rad]": AER[v]["CLa"], "induced k": AER[v]["k"], "Cd0": DRAG[v]["Cd0"],
               "CL max (aircraft)": st["CL max (aircraft)"], "stall speed [m/s]": TO.loc[v, "stall speed [m/s]"], "static margin [c]": st["static margin [c]"], "front share at trim": st["front share at trim"],
               "L/D at 16 m/s": c16["L/D"] if c16 is not None and c16["feasible"] else np.nan, "power at 16 m/s [W]": c16["electrical [W]"] if c16 is not None and c16["feasible"] else np.nan,
               "endurance best [min]": f["endurance [min]"].max(), "at [m/s]": f["endurance [min]"].idxmax(), "elevator trim at 16 [deg]": c16["elevator trim [deg]"] if c16 is not None and c16["feasible"] else np.nan,
               "ground roll [m]": TO.loc[v, "ground roll [m]"], "hover pitch [rad/s²]": hx.iloc[1]["angular acceleration [rad/s²]"], "hover heading [rad/s²]": hx.iloc[0]["angular acceleration [rad/s²]"],
               "spar reserve": SPAR.loc[v, "spar reserve"], "transitions passed": f"{VER[v]['passes']} of {VER[v]['of']}", "fall out of the hover, 3 s [m]": FALL.loc[f"{v}, 3.0 s", "height loss [m]"],
               "balloon into the hover, 3 s [m]": MAT.loc[(v, "back", 3.0, 0.0), "height gain [m]"], "bracing": "none"}
CMP = pd.DataFrame(rows).T
if BIP:
    b = BIP["variants"].get(BIP["main"], {})
    CMP.loc["Kestrel-4Q-2 (notebook 43)"] = {"mass [kg]": b.get("take-off mass [kg]"), "Cd0": b.get("Cd0"), "static margin [c]": b.get("static margin [chords]"),
                                              "power at 16 m/s [W]": b.get("power at 16 m/s [W]", b.get("cruise power [W]")), "endurance best [min]": b.get("endurance [min]"),
                                              "hover pitch [rad/s²]": BIP["hover pitch [rad/s²]"], "hover heading [rad/s²]": BIP["hover heading [rad/s²]"], "transitions passed": b.get("transitions passed"),
                                              "bracing": b.get("bracing", "faired posts"), "wing area [m²]": 0.28, "span [m]": 1.0}
display(CMP.T)
CMP.to_csv(OUT / "comparison.csv")
P2 = kestrel.resolve(dict(elevons=True, strut_fairing=True, bracing="posts"))
FOUR = {"Kestrel-4Q-2 (biplane, faired posts)": kt.make_vehicle(P2, faired=True, name="4Q-2")}
FOUR.update({v: dataclasses.replace(veh, pol=veh.aircraft_polar) for v, veh in VEH.items() if v != "Q4T1-10 (flat)"})
fig, axs = plt.subplots(1, 3, figsize=(17, 4.8))
DRAG4 = kd.drag_vs_speed(FOUR, speeds=np.arange(9.0, 40.5, 0.5), axs=axs); plt.tight_layout(); plt.show()
display(DRAG4.round(2))
print(f"§ 7 in {time.time() - T0:.0f} s")

## 8. Findings

Read with the numbers above (they are recomputed on every run; the sentences below are the brief's claims checked
against them):

- **Munk holds in the lattice.** The flat tandem's induced-drag factor is ~1.4 × the staggered tandem's, which equals
  the biplane's: stagger buys no induced drag, the gap does. What the tandem buys instead is **trim and control**:
  the rear wing is the tail, the sections need no reflex, the elevator sits 2.3 chords behind the CG.
- **The front wing stalls first** by construction: the CG 10 % of the stagger ahead of the neutral point puts two
  thirds of the lift on it; the aircraft's CL max is ~0.98 on the total area — the biplane's figure, not better: the
  stall speed and the ground roll are the biplane's too. The nose-down at the stall is what makes the fall out of
  the hover benign.
- **The pull-up into the hover gains height** — the tandem's clean, high-slope airframe balloons more than the
  biplane did; a descent into the transition and a slower schedule bring it back toward the datum. The way out,
  the direction the brief allows to fall, loses little in the model.
- **P against N** cannot be separated by this lattice (its wake is straight): the argument that the low rear wing of
  P flies into the front's wake at high α stands as an argument until the high-α CFD chapter measures it. The
  transition model sees the two as the same aircraft.
- **Span 1.2 m** is the honest lever: a few per cent of L/D, a lower stall speed, for 50 g and a spar still in reserve.
- **The flat tandem** loses on induced drag and has no pitch lever in the hover; its allocation matrix is rank 3.

The next steps are the plan's: the CFD chapters (the sweep per variant, the high-α cases where P's elevator lives, the
transition points with the rotor disks) and the FEA of the spar in its pod socket, as a book `kestrel-tandem` (code T).

In [ ]:
def plain(x):
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    return x
rows_ = lambda df: {" | ".join(map(str, k)) if isinstance(k, tuple) else str(k): {kk: plain(vv) for kk, vv in r.items()} for k, r in df.T.to_dict().items()}
DESIGN = {"source": "46_kestrel_tandem", "main": MAIN, "biplane_reference": BIP["source"] if BIP else None, "variants": ktd.VARIANTS,
          "layout": {v: {k: plain(x) for k, x in Lv.items() if isinstance(x, (int, float))} for v, Lv in LAY.items()},
          "mass": {v: {k: plain(x) for k, x in c.items() if k != "I"} for v, c in CI.items()}, "aero": {v: {k: plain(x) for k, x in a.items()} for v, a in AER.items()},
          "stall_rule": rows_(STALL.astype(object)), "spars": rows_(SPAR), "elevator": rows_(ELEV), "takeoff": rows_(TO.astype(object)), "transitions": rows_(MAT[cols]), "verdict": VER,
          "pull_up_remedies": rows_(REM[cols]), "fall_out_of_hover": rows_(FALL), "comparison": rows_(CMP.astype(object)), "drag_vs_speed": rows_(DRAG4)}
target = OUT / "kestrel_tandem_design.json" if DEBUG else Path("../designs/data/kestrel_tandem_design.json")
target.write_text(json.dumps(DESIGN, indent=1, default=plain))
print("design data:", target)

## 9. The dossier

This notebook's numbers as JSON (`designs/data/reports/46_kestrel_tandem.json`) and Markdown, stitched into the Kestrel dossier.

In [ ]:
import kestrel_report as kr
REPORT = {"variants": pd.DataFrame(ktd.VARIANTS).T, "layout": pd.DataFrame({v: {k: Lv[k] for k in keys} for v, Lv in LAY.items()}).T, "masses": pd.DataFrame({v: ks.groups(t) for v, t in TAB.items()}),
          "spars": SPAR, "lattice": A, "Munk": MUNK, "stall rule": STALL, "drag areas": pd.DataFrame({v: {k: x * 1e4 for k, x in d["parts_m2"].items()} for v, d in DRAG.items()}).T,
          "elevator": ELEV, "hover authority": {v: h for v, h in HOV.items()}, "cruise tables": {v: df.loc[[s for s in (12.0, 16.0, 20.0) if s in df.index]] for v, df in CRU.items()},
          "horizontal start": TO, "transitions": MAT[cols], "verdict": VER, "pull-up remedies": REM[cols], "the fall out of the hover": FALL, "against Kestrel-4Q-2 and between the variants": CMP,
          "drag vs speed": DRAG4, "runtime [s]": {"total": time.time() - T0}}
EXP = kr.export("46_kestrel_tandem", "Kestrel-Q4T — the tandem-wing Kestrel (Q4T1N-10 against Q4T1P-10, Q4T1N-12 and Kestrel-4Q-2)", REPORT,
                meta={"main": MAIN, "biplane_reference": BIP["source"] if BIP else None, "debug": DEBUG})
print(f"runtime {time.time() - T0:.0f} s")